# Stage 2A — Chunking Preprocessed Documents

**What this does:**
- Loads each preprocessed JSON from `MyDrive/L_P/docs/`
- Tokenizes every sentence using InLegalBERT tokenizer
- Splits long sentences into overlapping chunks of 128 tokens
- Saves chunked data to `MyDrive/L_P/chunks/doc_XXXXX.json`

**Why chunking?**
- InLegalBERT has a hard limit of 512 tokens per input
- Long sentences get silently cut off without chunking
- Chunking + averaging preserves the full sentence meaning

**Flow:**
```
L_P/docs/doc_XXXXX.json        ← preprocessed (Stage 1)
        ↓  THIS NOTEBOOK
L_P/chunks/doc_XXXXX.json      ← chunked (Stage 2A)
        ↓  Stage 2B (next)
L_P/embeddings/doc_XXXXX.pt    ← embedded
        ↓  Stage 3
L_P/graphs/doc_XXXXX.pt        ← graph for GNN
```

---
⚠️ Runtime: CPU is fine for chunking — no GPU needed here

In [1]:
# ── CELL 1 ── Install dependencies
!pip install transformers tqdm --quiet
print('✓ Done')

✓ Done


In [3]:
# ── CELL 2 ── Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
print('✓ Drive mounted')

Mounted at /content/drive
✓ Drive mounted


In [4]:
# ── CELL 4 ── Load InLegalBERT tokenizer
# Only the tokenizer is needed here — no model, no GPU
import transformers, logging
transformers.logging.set_verbosity_error()
logging.getLogger('transformers').setLevel(logging.ERROR)

from transformers import AutoTokenizer

MODEL_NAME = 'law-ai/InLegalBERT'
print(f'Loading tokenizer: {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer loaded')
print(f'Vocab size    : {tokenizer.vocab_size:,}')
print(f'Max length    : {tokenizer.model_max_length}')
print()
print('NOTE: sentences longer than 512 tokens will be split into')
print('overlapping chunks of 128 tokens each in Cell 6.')
print('This is expected and correct — no action needed.')


Loading tokenizer: law-ai/InLegalBERT...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/516 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded
Vocab size    : 30,522
Max length    : 512

NOTE: sentences longer than 512 tokens will be split into
overlapping chunks of 128 tokens each in Cell 6.
This is expected and correct — no action needed.


In [11]:
# ── CELL 5 ── Diagnostic — check token lengths across 500 docs
import json, os
import transformers
import logging

# Define paths and constants (assuming these would be defined earlier in a complete notebook)
DRIVE_ROOT = '/content/drive/MyDrive/L_P'
DOCS_DIR   = os.path.join(DRIVE_ROOT, 'docs')
CHUNKS_DIR = os.path.join(DRIVE_ROOT, 'chunks')
CKPT_FILE  = os.path.join(CHUNKS_DIR, 'chunking_checkpoint.txt')

CHUNK_SIZE = 256  # InLegalBERT max tokens is 512, but we chunk into 128 for finer granularity
OVERLAP    = 64   # Overlap between chunks

# Ensure output directory exists
os.makedirs(CHUNKS_DIR, exist_ok=True)

# List all preprocessed JSON files
all_jsons = sorted([f for f in os.listdir(DOCS_DIR) if f.endswith('.json')])
print(f'Found {len(all_jsons):,} preprocessed documents.')

# Suppress the '> 512 tokens' warning — we KNOW sentences can be long.
# That is exactly WHY we are chunking. The warning is informational only.
transformers.logging.set_verbosity_error()
logging.getLogger('transformers').setLevel(logging.ERROR)

sample_files = all_jsons[:500]
lengths = []

for fname in sample_files:
    with open(os.path.join(DOCS_DIR, fname)) as f:
        doc = json.load(f)
    for s in doc.get('sentences', []):
        # truncation=False  → get ALL tokens, no silent cutting
        # add_special_tokens=True → includes [CLS] and [SEP] in the count
        tokens = tokenizer(
            s['text'],
            truncation=False,
            add_special_tokens=True
        )['input_ids']
        lengths.append(len(tokens))

lengths.sort()
total = len(lengths)

print(f'Sentences checked       : {total:,}')
print(f'Min tokens              : {min(lengths)}')
print(f'Max tokens              : {max(lengths)}')
print(f'Mean tokens             : {sum(lengths)//total}')
print(f'Median tokens           : {lengths[total//2]}')
print()
print(f'Fit in 128 tokens  (<= 128): {sum(1 for l in lengths if l <= 128)/total*100:.1f}%')
print(f'Need chunking      (> 128) : {sum(1 for l in lengths if l > 128)/total*100:.1f}%')
print(f'Over 256 tokens    (> 256) : {sum(1 for l in lengths if l > 256)/total*100:.1f}%')
print(f'Over 512 tokens    (> 512) : {sum(1 for l in lengths if l > 512)/total*100:.1f}%')
print()
print('→ Sentences > 128 tokens will be split into overlapping chunks')
print('→ Sentences > 512 tokens would crash the model without chunking')
print('→ Your chunking logic (Cell 6) handles ALL of these correctly')


Found 5,000 preprocessed documents.
Sentences checked       : 84,281
Min tokens              : 6
Max tokens              : 811
Mean tokens             : 46
Median tokens           : 37

Fit in 128 tokens  (<= 128): 97.1%
Need chunking      (> 128) : 2.9%
Over 256 tokens    (> 256) : 0.3%
Over 512 tokens    (> 512) : 0.0%

→ Sentences > 128 tokens will be split into overlapping chunks
→ Sentences > 512 tokens would crash the model without chunking
→ Your chunking logic (Cell 6) handles ALL of these correctly


In [12]:
# ── CELL 5 ── Diagnostic — check token lengths across 500 docs
# This tells us how many sentences actually need chunking
import json

sample_files = all_jsons[:500]
lengths = []

for fname in sample_files:
    with open(os.path.join(DOCS_DIR, fname)) as f:
        doc = json.load(f)
    for s in doc.get('sentences', []):
        tokens = tokenizer(
            s['text'],
            truncation=False,
            add_special_tokens=True
        )['input_ids']
        lengths.append(len(tokens))

lengths.sort()
total = len(lengths)

print(f'Sentences checked       : {total:,}')
print(f'Min tokens              : {min(lengths)}')
print(f'Max tokens              : {max(lengths)}')
print(f'Mean tokens             : {sum(lengths)//total}')
print(f'Median tokens           : {lengths[total//2]}')
print()
print(f'% fitting in 128 tokens : {sum(1 for l in lengths if l <= 128) / total * 100:.1f}%')
print(f'% over 128 tokens       : {sum(1 for l in lengths if l > 128) / total * 100:.1f}%')
print(f'% over 256 tokens       : {sum(1 for l in lengths if l > 256) / total * 100:.1f}%')
print(f'% over 512 tokens       : {sum(1 for l in lengths if l > 512) / total * 100:.1f}%')
print()
print('→ Chunking is needed for all sentences > 128 tokens')

Sentences checked       : 84,281
Min tokens              : 6
Max tokens              : 811
Mean tokens             : 46
Median tokens           : 37

% fitting in 128 tokens : 97.1%
% over 128 tokens       : 2.9%
% over 256 tokens       : 0.3%
% over 512 tokens       : 0.0%

→ Chunking is needed for all sentences > 128 tokens


In [13]:
# ── CELL 6 ── Define chunking functions
import json

def chunk_sentence(text, tokenizer, chunk_size, overlap):
    """
    Split one sentence into overlapping token chunks.

    Example with chunk_size=6, overlap=2:
    tokens = [A, B, C, D, E, F, G, H]
    chunk1 = [A, B, C, D, E, F]
    chunk2 =          [E, F, G, H]   ← overlaps by 2

    Returns list of chunk dicts:
    [
      {'chunk_id': 0, 'text': '...', 'token_ids': [...], 'num_tokens': 128},
      {'chunk_id': 1, 'text': '...', 'token_ids': [...], 'num_tokens': 64},
    ]
    """
    # Tokenize without truncation to get ALL tokens
    encoded = tokenizer(
        text,
        truncation=False,
        add_special_tokens=False  # we add [CLS] and [SEP] per chunk below
    )
    token_ids = encoded['input_ids']

    # If sentence fits in one chunk — no splitting needed
    # Reserve 2 tokens for [CLS] and [SEP]
    effective_size = chunk_size - 2

    if len(token_ids) <= effective_size:
        chunk_text = tokenizer.decode(token_ids, skip_special_tokens=True)
        return [{
            'chunk_id':   0,
            'text':       chunk_text,
            'token_ids':  [tokenizer.cls_token_id] + token_ids + [tokenizer.sep_token_id],
            'num_tokens': len(token_ids) + 2,
            'is_chunked': False
        }]

    # Split into overlapping chunks
    stride  = effective_size - overlap
    chunks  = []
    start   = 0
    chunk_id = 0

    while start < len(token_ids):
        end        = min(start + effective_size, len(token_ids))
        chunk_toks = token_ids[start:end]
        chunk_text = tokenizer.decode(chunk_toks, skip_special_tokens=True)

        chunks.append({
            'chunk_id':   chunk_id,
            'text':       chunk_text,
            'token_ids':  [tokenizer.cls_token_id] + chunk_toks + [tokenizer.sep_token_id],
            'num_tokens': len(chunk_toks) + 2,
            'is_chunked': True
        })

        if end == len(token_ids):
            break
        start    += stride
        chunk_id += 1

    return chunks


def chunk_document(doc, tokenizer, chunk_size, overlap):
    """
    Process all sentences in a document.
    Returns chunked document dict ready to save.
    """
    chunked_sentences = []
    total_chunks      = 0
    chunked_count     = 0

    for sent in doc.get('sentences', []):
        chunks = chunk_sentence(
            sent['text'], tokenizer, chunk_size, overlap
        )
        total_chunks += len(chunks)
        if len(chunks) > 1:
            chunked_count += 1

        chunked_sentences.append({
            'sent_id':      sent['id'],
            'section':      sent['section'],
            'original_text': sent['text'],
            'num_chunks':   len(chunks),
            'chunks':       chunks          # list of chunk dicts
        })

    return {
        'doc_id':           doc['doc_id'],
        'filename':         doc.get('filename', ''),
        'entities':         doc.get('entities', {}),
        'num_sentences':    len(chunked_sentences),
        'total_chunks':     total_chunks,
        'sentences_chunked': chunked_count,  # how many needed splitting
        'sentences':        chunked_sentences
    }


def load_checkpoint(path):
    if not os.path.exists(path):
        return set()
    with open(path) as f:
        done = {l.strip() for l in f if l.strip()}
    print(f'  Checkpoint: {len(done):,} docs already done — skipping')
    return done


def save_checkpoint(path, doc_id):
    with open(path, 'a') as f:
        f.write(doc_id + '\n')


print('✓ Chunking functions defined')

✓ Chunking functions defined


In [14]:
# ── CELL 7 ── TEST on 3 documents
import json

test_files = all_jsons[:3]
print(f'Testing on {len(test_files)} documents...\n')

for fname in test_files:
    path = os.path.join(DOCS_DIR, fname)
    with open(path) as f:
        doc = json.load(f)

    chunked = chunk_document(doc, tokenizer, CHUNK_SIZE, OVERLAP)

    print(f'  {fname}')
    print(f'    Sentences        : {chunked["num_sentences"]}')
    print(f'    Total chunks     : {chunked["total_chunks"]}')
    print(f'    Sentences split  : {chunked["sentences_chunked"]}'
          f' ({chunked["sentences_chunked"]/max(chunked["num_sentences"],1)*100:.1f}%)')
    print()

    # Show first sentence that was chunked
    for s in chunked['sentences']:
        if s['num_chunks'] > 1:
            print(f'    Example chunked sentence:')
            print(f'      Original : {s["original_text"][:80]}...')
            print(f'      Chunks   : {s["num_chunks"]}')
            for c in s['chunks']:
                print(f'        Chunk {c["chunk_id"]}: {c["num_tokens"]} tokens | {c["text"][:60]}...')
            break
    print()

print('✓ Test passed — ready for full run')

Testing on 3 documents...

  doc_00000.json
    Sentences        : 2610
    Total chunks     : 2616
    Sentences split  : 4 (0.2%)

    Example chunked sentence:
      Original : Gopalan vs The State Of Madras.Union Of India: ... on 19 May, 1950 RF 1954 SC 72...
      Chunks   : 4
        Chunk 0: 256 tokens | gopalan vs the state of madras. union of india :... on 19 ma...
        Chunk 1: 256 tokens | ##c1047 ( 18 ) f 1963 sc1295 ( 15, 31 ) f 1964 sc 381 ( 54 )...
        Chunk 2: 256 tokens | 7, 9 ) rf 1973 sc 106 ( 105 ) o 1973 sc1425 ( 7, 18, 25, 27,...
        Chunk 3: 88 tokens | ( 89 ) r 1978 sc 215 ( 67 ) d 1978 sc 489 ( 1, 9 ) e & r 197...

  doc_00001.json
    Sentences        : 58
    Total chunks     : 58
    Sentences split  : 0 (0.0%)


  doc_00002.json
    Sentences        : 374
    Total chunks     : 376
    Sentences split  : 2 (0.5%)

    Example chunked sentence:
      Original : Held, per KANIA C.J., FAZL ALl, PATANJALI SASTRI and DAS JJ.--(i) that a house o...
   

In [15]:
# ── CELL 8 ── FULL RUN — chunk all 5,000 documents
#
# ✅ Resumable  — if Colab disconnects, just re-run this cell
# ⏱ Time       — ~20–40 min on CPU for 5,000 docs
# 💾 Output     — one chunked JSON per doc in MyDrive/L_P/chunks/

import json, os, time
from tqdm import tqdm

done  = load_checkpoint(CKPT_FILE)
to_do = [f for f in all_jsons if f.replace('.json', '') not in done]

print(f'Total   : {len(all_jsons):,}')
print(f'Done    : {len(done):,}')
print(f'To do   : {len(to_do):,}')
print('=' * 50)

n_ok   = 0
n_err  = 0
t0     = time.time()

total_sentences = 0
total_chunks    = 0
total_split     = 0

for fname in tqdm(to_do, desc='Chunking', unit='doc'):
    doc_id    = fname.replace('.json', '')
    in_path   = os.path.join(DOCS_DIR,   fname)
    out_path  = os.path.join(CHUNKS_DIR, fname)  # same filename, different folder

    try:
        # Load preprocessed JSON
        with open(in_path) as f:
            doc = json.load(f)

        # Chunk the document
        chunked = chunk_document(doc, tokenizer, CHUNK_SIZE, OVERLAP)

        # Save chunked JSON
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(chunked, f, ensure_ascii=False)
            # Note: no indent=2 to keep file size small

        total_sentences += chunked['num_sentences']
        total_chunks    += chunked['total_chunks']
        total_split     += chunked['sentences_chunked']

        n_ok += 1
        save_checkpoint(CKPT_FILE, doc_id)

    except Exception as e:
        n_err += 1
        print(f'\n  ERROR on {fname}: {e}')
        save_checkpoint(CKPT_FILE, doc_id)

elapsed = time.time() - t0

print('=' * 50)
print(f'✓ DONE')
print(f'  Successful          : {n_ok:,}')
print(f'  Errors              : {n_err:,}')
print(f'  Time                : {elapsed/60:.1f} min')
print(f'  Speed               : {n_ok/elapsed:.2f} docs/sec')
print()
print(f'  Total sentences     : {total_sentences:,}')
print(f'  Total chunks        : {total_chunks:,}')
print(f'  Sentences split     : {total_split:,} ({total_split/max(total_sentences,1)*100:.1f}%)')
print(f'  Avg chunks/sentence : {total_chunks/max(total_sentences,1):.2f}')
print()
print(f'  Saved to            : {CHUNKS_DIR}')

Total   : 5,000
Done    : 0
To do   : 5,000


Chunking: 100%|██████████| 5000/5000 [08:26<00:00,  9.88doc/s]

✓ DONE
  Successful          : 5,000
  Errors              : 0
  Time                : 8.4 min
  Speed               : 9.88 docs/sec

  Total sentences     : 792,238
  Total chunks        : 793,924
  Sentences split     : 1,529 (0.2%)
  Avg chunks/sentence : 1.00

  Saved to            : /content/drive/MyDrive/L_P/chunks


In [16]:
# ── CELL 9 ── Verify output
import json, os

chunk_files = sorted([f for f in os.listdir(CHUNKS_DIR) if f.endswith('.json')])
print(f'Total chunked files : {len(chunk_files):,}')
print()

# Inspect first file
if chunk_files:
    with open(os.path.join(CHUNKS_DIR, chunk_files[0])) as f:
        d = json.load(f)

    print(f'Sample file          : {chunk_files[0]}')
    print(f'doc_id               : {d["doc_id"]}')
    print(f'num_sentences        : {d["num_sentences"]}')
    print(f'total_chunks         : {d["total_chunks"]}')
    print(f'sentences_split      : {d["sentences_chunked"]}')
    print()

    # Show structure of first sentence
    s = d['sentences'][0]
    print(f'First sentence:')
    print(f'  sent_id      : {s["sent_id"]}')
    print(f'  section      : {s["section"]}')
    print(f'  num_chunks   : {s["num_chunks"]}')
    print(f'  original_text: {s["original_text"][:80]}...')
    print(f'  chunk 0 text : {s["chunks"][0]["text"][:80]}...')
    print(f'  chunk 0 toks : {s["chunks"][0]["num_tokens"]}')
    print()
    print('✓ Chunking complete!')
    print('✓ Next step: Run Stage 2B — Embedding the chunks with InLegalBERT')

Total chunked files : 5,000

Sample file          : doc_00000.json
doc_id               : doc_00000
num_sentences        : 2610
total_chunks         : 2616
sentences_split      : 4

First sentence:
  sent_id      : 0
  section      : PREAMBLE
  num_chunks   : 1
  original_text: Gopalan vs The State Of Madras.Union Of India: ... on 19 May, 1950 A.K....
  chunk 0 text : gopalan vs the state of madras. union of india :... on 19 may, 1950 a. k....
  chunk 0 toks : 30

✓ Chunking complete!
✓ Next step: Run Stage 2B — Embedding the chunks with InLegalBERT
